# 拣金：在 Colab 上跑 TOPReward（Qwen3-VL-4B）

TOPReward 是第二个视频评判，和 Robometer 互相独立。这次用 4B（上次 8B 加载时把内存撑爆了），直接加载到显卡。

分两部分，各自打包下载：
- **第一部分：考卷 A**（dev 103 对、test 201 对，共 608 条），视频约 1.6 GB。
- **第二部分（可选）：RoboVAD**（test 629、train 367，共 996 条），要下 6.86 GB。第一部分跑完还有时间再跑。

**怎么跑**
1. 运行时类型选 **T4 GPU**。
2. 把本机新打的 `dist/jianjin-0.1.0-py3-none-any.whl` 拖进 `/content/`（同名，内容是新的）。
3. 用左边「文件」面板的 Drive 按钮先挂好 Google Drive。
4. 运行第 1 到 7 格。第 4 格下载约 9 GB 权重，有进度条；第 5 格加载，会打印显存和内存；第 7 格冒烟 2 对，会打印每条耗时、原描述和换后描述的「True」概率，以及全量预计时间。**把第 5、7 格的输出贴给 Claude 看一眼再继续。**
5. 运行第 8 格（考卷 A 全量），再运行第 9 格下载 `topreward_exam_a.zip`。
6. 有时间就运行第 10 到 12 格（RoboVAD），第 12 格下载 `topreward_robovad.zip`。
7. 两个压缩包都放到项目的 `C:\Claude_code\jianjin\cache\` 下解压（包里自带 `topreward/` 这层目录），告诉 Claude。

断了就从第 1 格重跑，已经算完的会直接跳过。


In [ ]:
# 1. GPU、装依赖（lerobot 锁 0.6.1：适配器用到了它的 TOPReward 编码步骤）
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "lerobot[topreward]==0.6.1" psutil 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps /content/jianjin-0.1.0-py3-none-any.whl 2>&1 | tail -2
import importlib.metadata as md
print("lerobot", md.version("lerobot"), "transformers", md.version("transformers"), "jianjin", md.version("jianjin"))

In [ ]:
# 2. 缓存放 Google Drive（和前两次同一个目录，文件在 topreward/ 下，不冲突）
import os
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/jianjin_cache"
os.makedirs(CACHE, exist_ok=True)
print("缓存目录", CACHE)

In [ ]:
# 3. 没挂 Drive 时的缓存位置
try:
    CACHE
except NameError:
    CACHE = "/content/cache"
    os.makedirs(CACHE, exist_ok=True)
print("缓存目录", CACHE)

In [ ]:
# 4. 下载 Qwen3-VL-4B 权重（约 9 GB，带进度条）；下好了再运行这格会秒过
import psutil
from huggingface_hub import snapshot_download
def ram():
    m = psutil.virtual_memory()
    return f"内存 已用 {(m.total - m.available) / 1e9:.1f} / {m.total / 1e9:.1f} GB"
print(snapshot_download("Qwen/Qwen3-VL-4B-Instruct"))
print(ram())

In [ ]:
# 5. 直接加载到显卡（半精度）。正常的话显存约 9 GB，内存不会涨到顶
import gc, time, torch
gc.collect()
print("加载前", ram(), flush=True)
from jianjin.judges.topreward import TOPRewardJudge
J = TOPRewardJudge()
t = time.time()
J._load()
gc.collect()
print("加载完", f"{time.time() - t:.0f}s", J.version, ram(),
      f"显存 {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# 6. 考卷 A 的数据：关节数据和配置里那一路相机的视频（约 1.6 GB，有进度条）
CONFIG = "/content/exam_a.toml"
open(CONFIG, "w", encoding="utf-8").write('# 考卷 A 用的公开数据集。一个数据集只属于一个切分；授权照抄数据卡。\n# camera 是给视频评判用的那一路（俯视或侧视，看得到整个工作台）。\n# 开发集定阈值；测试集换了任务（分拣、乐高块）、换了机型（SO-101、双臂），只跑一次。\n[[dataset]]\nrepo = "lerobot/svla_so100_pickplace"   # 50 条，把方块放进盒子\nsplit = "dev"\nlicense = "apache-2.0"\ncamera = "observation.images.top"\n\n[[dataset]]\nrepo = "lerobot/svla_so100_stacking"    # 56 条，红块叠到蓝块上\nsplit = "dev"\nlicense = "apache-2.0"\ncamera = "observation.images.top"\n\n[[dataset]]\nrepo = "lerobot/svla_so100_sorting"     # 52 条，红蓝块分进左右两个盒子\nsplit = "test"\nlicense = "apache-2.0"\ncamera = "observation.images.top"\n\n[[dataset]]\nrepo = "lerobot/svla_so101_pickplace"   # 50 条，乐高块放进透明盒；robot_type 写的是 so100_follower\nsplit = "test"\nlicense = "apache-2.0"\ncamera = "observation.images.up"\n\n[[dataset]]\nrepo = "CoRL2026-CSI/SO101-DualArm-teleop_fold_towel_100epi"  # 100 条，双臂叠毛巾\nsplit = "test"\nlicense = "apache-2.0"\ncamera = "observation.images.left_top"\n')
from jianjin.eval.exam_a import load_config
from jianjin.eval.fetch import fetch
SPECS = load_config(CONFIG)
DATA = "/content/data"
for p in fetch(SPECS, DATA, with_videos=True):
    print(p.name, sum(1 for _ in p.rglob("*.mp4")), "个视频文件")

In [ ]:
# 7. 冒烟：dev 里前 2 对，看原描述和换后描述的「True」概率差多少，估算全量时间
from jianjin.eval.task_swap import swap_pairs
from jianjin.judges.cache import judge_episode
PAIRS = {s: swap_pairs(SPECS, DATA, s) for s in ["dev", "test"]}
print({s: len(p) for s, p in PAIRS.items()}, "对")
per = []
for orig, swapped, cam in PAIRS["dev"][:2]:
    for ep in (orig, swapped):
        torch.cuda.reset_peak_memory_stats()
        t = time.time()
        c = judge_episode(J, ep, cam, CACHE, sample_fps=2.0, max_side=336)
        per.append(time.time() - t)
        print(ep.id, f"{per[-1]:.1f}s", f"峰值 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
        print("  任务", ep.task)
        print("  True 概率", [round(p, 3) for p in c.success])
        print("  进度", [round(p, 2) for p in c.progress])
n = 2 * sum(len(p) for p in PAIRS.values())
avg = sum(per) / len(per)
print(f"考卷 A 全量 {n} 条预计 {avg * n / 3600:.1f} 小时；RoboVAD 996 条约 {avg * 996 / 3600:.1f} 小时")

In [ ]:
# 8. 考卷 A 全量：先 dev 再 test；断了重跑这一格，已算完的直接跳过
from tqdm.auto import tqdm
from jianjin.checks import NotApplicable
def run(jobs, desc):
    skipped = 0
    for ep, cam in tqdm(jobs, desc=desc):
        try:
            judge_episode(J, ep, cam, CACHE, sample_fps=2.0, max_side=336)
        except NotApplicable as e:
            skipped += 1; print("跳过", ep.id, e)
        except torch.OutOfMemoryError:
            skipped += 1; print("显存不够，跳过", ep.id)
            gc.collect(); torch.cuda.empty_cache()
        except RuntimeError as e:  # 半精度溢出之类
            skipped += 1; print("出错，跳过", ep.id, str(e)[:200])
    print(desc, "完成", len(jobs) - skipped, "跳过", skipped, flush=True)
for split, pairs in PAIRS.items():
    run([(ep, cam) for o, s, cam in pairs for ep in (o, s)], f"考卷 A {split}")

In [ ]:
# 9. 只打包考卷 A 的 TOPReward 缓存，压缩包里带 topreward/ 这一层目录
import zipfile
from pathlib import Path
from google.colab import files
PREFIXES = tuple(s.repo.replace("/", "__") for s in SPECS)
def pack(name, keep):
    src = Path(CACHE) / "topreward"
    picked = [f for f in src.iterdir() if keep(f.name)]
    with zipfile.ZipFile(f"/content/{name}", "w", zipfile.ZIP_DEFLATED) as z:
        for f in picked:
            z.write(f, f"topreward/{f.name}")
    print(name, len(picked), "个文件")
    files.download(f"/content/{name}")
pack("topreward_exam_a.zip", lambda n: n.startswith(PREFIXES))

## 第二部分（可选）：RoboVAD

要下 6.86 GB，跑 996 条。时间以第 7 格的估算为准。

In [ ]:
# 10. 下载 RoboVAD（带进度条，断了再运行会接着下），只解压左路视频和标注、关节数据
import urllib.request
URL = "https://zenodo.org/records/22754659/files/RoboVAD.zip?download=1"
ZIP = "/content/RoboVAD.zip"
if not os.path.exists("/content/RoboVAD/annotations.json"):
    total = int(urllib.request.urlopen(urllib.request.Request(URL, method="HEAD")).headers["Content-Length"])
    have = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
    if have < total:
        req = urllib.request.Request(URL, headers={"Range": f"bytes={have}-"})
        with urllib.request.urlopen(req) as r, open(ZIP, "ab") as f, \
                tqdm(total=total, initial=have, unit="B", unit_scale=True, desc="下载") as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    with zipfile.ZipFile(ZIP) as z:
        members = [m for m in z.infolist()
                   if m.filename.startswith("RoboVAD/") and not m.filename.endswith("_right.mp4")]
        for m in tqdm(members, desc="解压"):
            z.extract(m, "/content")
    os.remove(ZIP)
print(len(os.listdir("/content/RoboVAD/videos")), "个视频")

In [ ]:
# 11. RoboVAD 全量：先 test 再 train，左路相机；断了重跑这一格，已算完的直接跳过
import csv
from jianjin.ingest.robovad import read_robovad
ROOT = "/content/RoboVAD"
for split in ["test", "train"]:
    names = [r["episode_name"] for r in csv.DictReader(open(f"{ROOT}/splits/{split}.csv"))]
    run([(ep, "left") for ep in read_robovad(ROOT, names=names).episodes], f"RoboVAD {split}")

In [ ]:
# 12. 只打包 RoboVAD 的 TOPReward 缓存
pack("topreward_robovad.zip", lambda n: not n.startswith(PREFIXES))